# Coffee17 B0/C2 checkpoint recovery V1 — dataset selector revision V2
Exactly 2 architectures × 5 folds, seed 42. User-authorized recovery only. No outer test. All weights and logs retained. Attach original Coffee17 dataset; Internet ON; GPU required. For a resumed session, attach the previous recovery output, not the master/focused compact ZIPs.

In [ ]:
import os, sys, json, shutil, subprocess, tempfile, zipfile
from pathlib import Path
import torch
PIN = "12cc8e614c63968cd2df69db42490f6f3da19450"
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
PROJECT = WORK / "coffee17-b0-c2-recovery-v1"
SCRATCH = Path("/tmp/coffee17_b0_c2_recovery_v1_data")
MANIFESTS = PROJECT / "manifests"
assert INPUT.is_dir() and WORK.is_dir(), "Run on Kaggle"
if not torch.cuda.is_available():
    raise RuntimeError("Enable GPU; no CPU fallback")
print("RECOVERY ONLY: B0, C2 | FOLDS 1-5 | SEED 42")
print("OUTER TEST: LOCKED")
print("CHECKPOINT DELETION: DISABLED")
print("PIN:", PIN, "GPU:", torch.cuda.get_device_name(0))
REPO = Path(tempfile.mkdtemp(prefix="coffee17-recovery-src-")) / "repo"
subprocess.run(["git","clone","--quiet","--no-checkout","https://github.com/ediprin/coffee-bean-classification.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--quiet","--detach",PIN],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(REPO/"requirements/preprocessing-study.txt")],check=True)
ENV=dict(os.environ)
ENV["PYTHONPATH"]=str(REPO/"src")+os.pathsep+ENV.get("PYTHONPATH","")
sys.path.insert(0,str(REPO/"src"))
RUNNER=REPO/"src/bilinear_lmmd/experiments/run_c2_recovery_v1.py"
compile(RUNNER.read_text(),str(RUNNER),"exec")
# Restore only if target does not exist, never merge/overwrite partial runs.
prior=sorted(p for p in (INPUT/"notebooks").rglob(PROJECT.name) if p.is_dir() and (p/"recovery_authority.json").is_file()) if (INPUT/"notebooks").exists() else []
# Older Kaggle layouts may mount notebook outputs directly below /kaggle/input.
if not prior:
    prior=sorted(p for p in INPUT.rglob(PROJECT.name) if p.is_dir() and (p/"recovery_authority.json").is_file() and "datasets" not in p.relative_to(INPUT).parts)
if not PROJECT.exists() and len(prior)>1:
    raise RuntimeError("Multiple recovery outputs; attach exactly one")
if not PROJECT.exists() and len(prior)==1:
    shutil.copytree(prior[0],PROJECT)
    print("RESTORED:",prior[0])
PROJECT.mkdir(parents=True,exist_ok=True)
SCRATCH.mkdir(parents=True,exist_ok=True)


In [ ]:
from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.experiments.run_c2_recovery_v1 import verify_manifest

# Dataset selection only; never scan notebook output directories.
# Reuse the previously successful Focused C1/C2 dataset-tree discovery.
SLUG = "coffee-green-bean-with-17-defects-original"
DATASETS_INPUT = INPUT / "datasets"
preferred = [DATASETS_INPUT / "sujitraarw" / SLUG, INPUT / SLUG]
present = [p for p in preferred if p.is_dir()]
if len(present) > 1:
    raise RuntimeError(f"Multiple original dataset mounts: {present}")
if present:
    source = present[0]
    by_class = discover_directory_samples(source)
else:
    valid = []
    if DATASETS_INPUT.is_dir():
        roots = [DATASETS_INPUT] + sorted(p for p in DATASETS_INPUT.rglob("*") if p.is_dir())
        for root in roots:
            try:
                samples = discover_directory_samples(root)
            except (ValueError, FileNotFoundError):
                continue
            if len(samples) == 17 and sum(map(len, samples.values())) == 979:
                valid.append((root, samples))
    deepest = [(root, samples) for root, samples in valid
               if not any(other != root and root in other.parents for other, _ in valid)]
    if len(deepest) != 1:
        print("KAGGLE INPUT MOUNTS:", [str(p) for p in sorted(INPUT.iterdir())])
        if DATASETS_INPUT.is_dir():
            print("DATASET OWNERS:", [str(p) for p in sorted(DATASETS_INPUT.iterdir())])
            print("DATASET MOUNTS:", [str(p) for p in sorted(DATASETS_INPUT.glob("*/*")) if p.is_dir()])
        raise RuntimeError(
            "Cannot identify exactly one original Coffee17 dataset in the dataset input tree. "
            f"Valid roots: {[str(root) for root, _ in deepest]}. "
            "Add Input: sujitraarw/coffee-green-bean-with-17-defects-original (not a notebook output)."
        )
    source, by_class = deepest[0]

raw_count = sum(map(len, by_class.values()))
if len(by_class) != 17 or raw_count != 979:
    raise RuntimeError(f"Expected 979 images / 17 classes; got {raw_count} / {len(by_class)}")
print("RECOVERY DATASET SELECTOR: V2")
print("USING DATASET INPUT ONLY:", source)
print("Coffee17 selected:", raw_count, "images /", len(by_class), "classes")
ARCHIVE=SCRATCH/"original.zip"
CANONICAL=SCRATCH/"canonical"
PROV=SCRATCH/"provenance"
with zipfile.ZipFile(ARCHIVE,"w",compression=zipfile.ZIP_STORED) as z:
    for cls,paths in sorted(by_class.items()):
        for p in sorted(paths):
            info=zipfile.ZipInfo(f"{cls}/{p.name}",date_time=(1980,1,1,0,0,0))
            info.compress_type=zipfile.ZIP_STORED
            info.external_attr=0o644<<16
            z.writestr(info,p.read_bytes())
prov=audit_coffee17_provenance(ARCHIVE,PROV,canonical_root=CANONICAL)
assert prov["decision"]=="PASS"
# Generate fresh manifests in a temporary directory, compare to any prior copy.
fresh=Path(tempfile.mkdtemp(prefix="coffee17-recovery-manifests-"))
summary=prepare_preprocessing_folds(CANONICAL,PROV/"coffee17_provenance.json",fresh,folds=5,seed=42,validation_ratio=.10)
verify_manifest(json.loads((fresh/"clean_manifest.json").read_text()),json.loads((fresh/"fold_manifest.json").read_text()))
MANIFESTS.mkdir(exist_ok=True)
for p in sorted(fresh.glob("*.json")):
    dest=MANIFESTS/p.name
    if dest.exists() and json.loads(dest.read_text())!=json.loads(p.read_text()):
        raise RuntimeError(f"Recovered manifest differs: {dest}")
    if not dest.exists(): shutil.copy2(p,dest)
for name in ("coffee17_raw_manifest.json","coffee17_provenance.json"):
    dest=MANIFESTS/name
    if dest.exists() and json.loads(dest.read_text())!=json.loads((PROV/name).read_text()):
        raise RuntimeError(f"Provenance changed: {dest}")
    if not dest.exists(): shutil.copy2(PROV/name,dest)
print("ALL FIVE HISTORICAL VALIDATION HASHES: MATCH")
print("CLEAN POPULATION:",summary["clean_count"],"RAW:",summary["raw_count"])
print("No outer test materialized for inference.")


In [ ]:
# This is the only cell that runs training.
# The user authorized exactly this recovery; do not change arms or epochs.
cmd=[sys.executable,"-u","-m","bilinear_lmmd.experiments.run_c2_recovery_v1",
     "--config",str(REPO/"configs/frequency_screening/FOCUSED_EFFICIENCY_V1.yaml"),
     "--canonical",str(CANONICAL),"--manifests",str(MANIFESTS),
     "--output",str(PROJECT),"--scratch",str(SCRATCH/"development"),
     "--required-commit",PIN,"--authorize-recovery"]
with (PROJECT/"recovery.log").open("a",encoding="utf-8") as log:
    proc=subprocess.Popen(cmd,cwd=REPO,env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in proc.stdout:
        print(line,end="",flush=True)
        log.write(line); log.flush()
    if proc.wait()!=0:
        raise RuntimeError("Recovery stopped; inspect log. All artifacts preserved.")


In [ ]:
# Analysis packaging only; NEVER trains, deletes or moves weights.
import hashlib
summary_path=PROJECT/"analysis/recovery_v1.json"
assert summary_path.is_file(), "Recovery not complete"
summary=json.loads(summary_path.read_text())
assert summary["training_runs"]==10 and summary["outer_test_accessed"] is False
def sha(p):
    with p.open("rb") as f:
        return hashlib.file_digest(f,"sha256").hexdigest()
for fold in range(1,6):
    for arm in ("B0","C2"):
        d=PROJECT/"experiments"/f"fold_{fold}"/"seed42"/arm
        marker=json.loads((d/"completed.json").read_text())
        for name,expected in marker["files"].items():
            assert sha(d/name)==expected, (fold,arm,name)
        assert (d/"best.pt").is_file() and (d/"last.pt").is_file()
OUT=WORK/"coffee17-b0-c2-recovery-analysis-v1.zip"
files=sorted(p for p in PROJECT.rglob("*") if p.is_file() and p.suffix in {".json",".yaml",".csv",".log"} and not p.name.endswith(".tmp"))
# Write to new archive; if it already exists verify matching contents.
if OUT.exists():
    with zipfile.ZipFile(OUT) as z:
        assert z.testzip() is None
        assert set(z.namelist())=={str(p.relative_to(PROJECT)) for p in files}, "Existing ZIP differs; preserve and rename it"
        for p in files:
            assert hashlib.sha256(z.read(str(p.relative_to(PROJECT)))).hexdigest()==sha(p), "Existing ZIP differs; preserve and rename it"
else:
    tmp=OUT.with_suffix(".tmp")
    with zipfile.ZipFile(tmp,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for p in files: z.write(p,str(p.relative_to(PROJECT)))
    with zipfile.ZipFile(tmp) as z:
        assert z.testzip() is None
    tmp.replace(OUT)
print("UPLOAD:",OUT)
print("KEEP SAVED OUTPUT:",PROJECT)
print("20 BEST/LAST CHECKPOINTS VERIFIED AND RETAINED")
print("OUTER TEST: LOCKED")
print(json.dumps(summary["aggregate"],indent=2))
